# Wildfire Research — Raw Data Extraction
**Boundary:** Thailand  |  **Period:** 2001–2023

| # | Dataset | Source | Output |
|---|---------|--------|--------|
| 1 | Monthly Burned Area | MCD64A1 v6.1 | 276 rows — `year, month, burned_area_km2` |
| 2 | Annual Burned Area by LC Class | MCD12Q1 v6.1 + MCD64A1 | 23 rows — `year, lc01…lc17_burned_km2` |
| 3 | Climate | CHIRPS + MEI v2 + SPEI-3 | 276 rows — `year, month, chirps_mm, mei_v2, spei3` |

Output files are written to `./data_raw/`

## 0 · Setup

In [2]:
# !pip install earthengine-api scipy pandas numpy requests -q

import ee
import pandas as pd
import numpy as np
import requests
import os
from scipy.stats import fisk, norm

os.makedirs('data_raw', exist_ok=True)

# ── GEE init ──────────────────────────────────────────────────────────
# ee.Authenticate()   # uncomment + run once in a new environment
ee.Authenticate()
try:
    ee.Initialize(project="ee-sakda-451407")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project="ee-sakda-451407")

# ── Geometry & date constants ──────────────────────────────────────────
TH_GEOM  = (ee.FeatureCollection('FAO/GAUL/2015/level0')
              .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))
              .geometry())

CM_POINT = ee.Geometry.Point([98.9853, 18.7883])   # Chiang Mai city centre

YEARS  = list(range(2001, 2024))   # 23 years
MONTHS = list(range(1, 13))

print('GEE initialised  |  Thailand geometry loaded')

GEE initialised  |  Thailand geometry loaded


---
## 1 · MCD64A1 v6.1 — Monthly Burned Area (km²)
Collection: `MODIS/061/MCD64A1`  
Band: `BurnDate` — Julian day of burn (> 0 = confirmed burned pixel)  
Scale: 500 m  →  area calculated with `ee.Image.pixelArea()` for projection accuracy

In [3]:
def monthly_burned_area_km2(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, 'month')

    img = (ee.ImageCollection('MODIS/061/MCD64A1')
             .filterDate(start, end)
             .select('BurnDate')
             .first())

    # BurnDate > 0  →  confirmed burned pixel
    burned = img.gt(0)
    area   = ee.Image.pixelArea().divide(1e6).updateMask(burned)

    try:
        val = area.reduceRegion(
            reducer   = ee.Reducer.sum(),
            geometry  = TH_GEOM,
            scale     = 500,
            maxPixels = 1e13
        ).getInfo()
        return val.get('area') or 0
    except Exception as e:
        print(f'  [warn] {year}-{month:02d}: {e}')
        return 0

In [4]:
rows_ba = []
n, i   = len(YEARS) * 12, 0

for year in YEARS:
    for month in MONTHS:
        area = monthly_burned_area_km2(year, month)
        rows_ba.append({'year': year, 'month': month, 'burned_area_km2': round(area, 4)})
        i += 1
        print(f'\r[{i:>3}/{n}]  {year}-{month:02d}  {area:9.2f} km2', end='', flush=True)
    print()

df_ba = pd.DataFrame(rows_ba)
df_ba.to_csv('data_raw/mcd64a1_monthly_burned_area.csv', index=False)
print(f'\nRows: {len(df_ba)}   saved -> data_raw/mcd64a1_monthly_burned_area.csv')
df_ba.tail(6)

[ 12/276]  2001-12     966.24 km2
[ 24/276]  2002-12     435.87 km2
[ 36/276]  2003-12    1693.02 km2
[ 48/276]  2004-12    2464.54 km2
[ 60/276]  2005-12     396.22 km2
[ 72/276]  2006-12     447.24 km2
[ 84/276]  2007-12     401.41 km2
[ 96/276]  2008-12     243.82 km2
[108/276]  2009-12     346.42 km2
[120/276]  2010-12     342.51 km2
[132/276]  2011-12     551.31 km2
[144/276]  2012-12     509.23 km2
[156/276]  2013-12     444.87 km2
[168/276]  2014-12     741.43 km2
[180/276]  2015-12     591.70 km2
[192/276]  2016-12     273.23 km2
[204/276]  2017-12     573.26 km2
[216/276]  2018-12     658.42 km2
[228/276]  2019-12    1973.06 km2
[240/276]  2020-12     347.72 km2
[252/276]  2021-12     145.73 km2
[264/276]  2022-12     170.20 km2
[276/276]  2023-12     175.22 km2

Rows: 276   saved -> data_raw/mcd64a1_monthly_burned_area.csv


,year,month,burned_area_km2
270,2023,7,0.0000
271,2023,8,170.7368
272,2023,9,20.7105
273,2023,10,9.1289
274,2023,11,252.6067
275,2023,12,175.2247


---
## 2 · MCD12Q1 v6.1 — Annual Burned Area by LC_Type1
Collections: `MODIS/061/MCD12Q1` (LC_Type1, IGBP) + `MODIS/061/MCD64A1`  
Approach: build a **17-band stacked image** (one band per LC class, masked to burned pixels)
and run a single `reduceRegion` per year → only **23 GEE calls** total.

**Focus class:** `lc04` = Deciduous Broadleaf Forest

In [5]:
# IGBP LC_Type1 class labels
LC_NAMES = {
     1: 'Evergreen_Needleleaf_Forest',
     2: 'Evergreen_Broadleaf_Forest',
     3: 'Deciduous_Needleleaf_Forest',
     4: 'Deciduous_Broadleaf_Forest',   # primary focus
     5: 'Mixed_Forest',
     6: 'Closed_Shrubland',
     7: 'Open_Shrubland',
     8: 'Woody_Savanna',
     9: 'Savanna',
    10: 'Grassland',
    11: 'Permanent_Wetland',
    12: 'Cropland',
    13: 'Urban',
    14: 'Cropland_NatVeg_Mosaic',
    15: 'Snow_and_Ice',
    16: 'Barren',
    17: 'Water',
}


def annual_burned_by_lc_km2(year):
    """One GEE reduceRegion call -> burned area (km2) for all 17 LC_Type1 classes."""

    # Annual burned mask — pixel = 1 if burned in any month of the year
    burned = (ee.ImageCollection('MODIS/061/MCD64A1')
                .filterDate(f'{year}-01-01', f'{year}-12-31')
                .select('BurnDate')
                .map(lambda img: img.gt(0).rename('b'))
                .max())

    # Annual land-cover map (MCD12Q1 is released once per year)
    lc = (ee.ImageCollection('MODIS/061/MCD12Q1')
            .filterDate(f'{year}-01-01', f'{year+1}-01-01')
            .select('LC_Type1')
            .first())

    area_img = ee.Image.pixelArea().divide(1e6)

    # Stack 17 bands — each = pixel area where (lc == class) AND burned, else masked
    bands   = [area_img.updateMask(lc.eq(cls).And(burned)).rename(f'lc{cls:02d}')
               for cls in range(1, 18)]
    stacked = ee.Image.cat(bands)

    try:
        vals = stacked.reduceRegion(
            reducer   = ee.Reducer.sum(),
            geometry  = TH_GEOM,
            scale     = 500,
            maxPixels = 1e13
        ).getInfo()
    except Exception as e:
        print(f'  [warn] {year}: {e}')
        vals = {}

    row = {'year': year}
    for cls in range(1, 18):
        row[f'lc{cls:02d}_burned_km2'] = round(vals.get(f'lc{cls:02d}') or 0, 4)
    return row

In [6]:
rows_lc = []
for year in YEARS:
    row    = annual_burned_by_lc_km2(year)
    rows_lc.append(row)
    lc04   = row['lc04_burned_km2']
    lc02   = row['lc02_burned_km2']
    lc05   = row['lc05_burned_km2']
    print(f'{year}  lc04={lc04:8.2f} (DecidBroadleaf)  lc02={lc02:7.2f}  lc05={lc05:7.2f}')

df_lc = pd.DataFrame(rows_lc)

# Append human-readable header row as a CSV comment (saved separately)
df_lc.to_csv('data_raw/mcd12q1_burned_by_lc_annual.csv', index=False)
print(f'\nRows: {len(df_lc)}   saved -> data_raw/mcd12q1_burned_by_lc_annual.csv')

# LC class lookup table
lc_lookup = pd.DataFrame([
    {'col': f'lc{cls:02d}_burned_km2', 'class_id': cls, 'class_name': LC_NAMES[cls]}
    for cls in range(1, 18)
])
lc_lookup.to_csv('data_raw/lc_class_lookup.csv', index=False)

df_lc[['year', 'lc04_burned_km2', 'lc02_burned_km2', 'lc05_burned_km2']].head(6)

2001  lc04=  322.11 (DecidBroadleaf)  lc02= 259.05  lc05= 187.32
2002  lc04=  856.82 (DecidBroadleaf)  lc02= 748.91  lc05= 337.71
2003  lc04= 1465.46 (DecidBroadleaf)  lc02= 487.20  lc05= 603.69
2004  lc04= 2525.24 (DecidBroadleaf)  lc02=3075.28  lc05=1319.22
2005  lc04= 2464.02 (DecidBroadleaf)  lc02=1275.63  lc05= 709.74
2006  lc04= 1652.94 (DecidBroadleaf)  lc02= 551.02  lc05= 322.64
2007  lc04= 4385.34 (DecidBroadleaf)  lc02=2203.26  lc05=1116.90
2008  lc04= 1899.94 (DecidBroadleaf)  lc02= 881.70  lc05= 471.09
2009  lc04= 3129.46 (DecidBroadleaf)  lc02=1285.60  lc05= 474.25
2010  lc04= 2180.73 (DecidBroadleaf)  lc02=1225.33  lc05= 345.39
2011  lc04= 1636.59 (DecidBroadleaf)  lc02= 179.91  lc05= 105.17
2012  lc04= 2853.29 (DecidBroadleaf)  lc02=1140.21  lc05= 672.25
2013  lc04= 1568.53 (DecidBroadleaf)  lc02= 649.87  lc05= 200.14
2014  lc04= 3911.48 (DecidBroadleaf)  lc02=1217.77  lc05= 848.19
2015  lc04= 3180.59 (DecidBroadleaf)  lc02= 828.47  lc05= 741.62
2016  lc04= 2139.45 (Deci

,year,lc04_burned_km2,lc02_burned_km2,lc05_burned_km2
0,2001,322.1125,259.0482,187.3194
1,2002,856.8167,748.9086,337.7128
2,2003,1465.4563,487.1971,603.6899
3,2004,2525.2435,3075.2839,1319.2239
4,2005,2464.0249,1275.6251,709.7446
5,2006,1652.9379,551.0198,322.6407


---
## 3a · CHIRPS — Monthly Mean Rainfall (mm, Thailand average)
Collection: `UCSB-CHG/CHIRPS/MONTHLY`  |  Native resolution: ~0.05° (~5566 m)

In [9]:
def monthly_chirps_mm(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, 'month')
    # UCSB-CHG/CHIRPS/MONTHLY does not exist in GEE — aggregate from daily
    img = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
             .filterDate(start, end)
             .select('precipitation')
             .sum())   # sum daily values = monthly total (mm)
    try:
        val = img.reduceRegion(
            reducer   = ee.Reducer.mean(),
            geometry  = TH_GEOM,
            scale     = 5566,
            maxPixels = 1e13
        ).getInfo()
        rain = val.get('precipitation')
        return float(rain) if rain is not None else np.nan
    except Exception as e:
        print(f'  [warn] {year}-{month:02d}: {e}')
        return np.nan


rows_rain = []
for year in YEARS:
    for month in MONTHS:
        rain = monthly_chirps_mm(year, month)
        v    = round(rain, 4) if (rain is not None and not np.isnan(rain)) else np.nan
        rows_rain.append({'year': year, 'month': month, 'chirps_mm': v})
    print(f'CHIRPS {year} done')

df_rain = pd.DataFrame(rows_rain)
print(f'\nCHIRPS rows: {len(df_rain)}')
df_rain.head(6)

CHIRPS 2001 done
CHIRPS 2002 done
CHIRPS 2003 done
CHIRPS 2004 done
CHIRPS 2005 done
CHIRPS 2006 done
CHIRPS 2007 done
CHIRPS 2008 done
CHIRPS 2009 done
CHIRPS 2010 done
CHIRPS 2011 done
CHIRPS 2012 done
CHIRPS 2013 done
CHIRPS 2014 done
CHIRPS 2015 done
CHIRPS 2016 done
CHIRPS 2017 done
CHIRPS 2018 done
CHIRPS 2019 done
CHIRPS 2020 done
CHIRPS 2021 done
CHIRPS 2022 done
CHIRPS 2023 done

CHIRPS rows: 276


,year,month,chirps_mm
0,2001,1,30.0799
1,2001,2,13.7757
2,2001,3,124.5981
3,2001,4,55.7361
4,2001,5,222.8870
5,2001,6,192.6319


---
## 3b · MEI v2 — Multivariate ENSO Index (NOAA PSL)
Source: https://psl.noaa.gov/enso/mei/  
MEI v2 is computed for 12 overlapping **bi-monthly** seasons per year:  
DJ · JF · FM · MA · AM · MJ · JJ · JA · AS · SO · ON · ND  
→ assigned to `month` 1–12 as a standard monthly proxy in climate–fire studies.

In [10]:
def fetch_mei_v2(start_year=2001, end_year=2023):
    url  = 'https://psl.noaa.gov/enso/mei/data/meiv2.data'
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()

    rows = []
    for line in resp.text.strip().splitlines():
        parts = line.split()
        if len(parts) < 13:
            continue
        try:
            yr = int(float(parts[0]))   # skip header/text lines (e.g. "Row", year range)
        except ValueError:
            continue
        if not (start_year <= yr <= end_year):
            continue
        for m in range(1, 13):
            try:
                v = float(parts[m])
            except ValueError:
                v = np.nan
            rows.append({
                'year': yr, 'month': m,
                'mei_v2': round(v, 3) if (not np.isnan(v) and v > -900) else np.nan
            })
    return pd.DataFrame(rows)


df_mei = fetch_mei_v2()
print(f'MEI v2 rows: {len(df_mei)}')
df_mei.head(6)

MEI v2 rows: 276


,year,month,mei_v2
0,2001,1,-0.84
1,2001,2,-0.86
2,2001,3,-0.78
3,2001,4,-0.55
4,2001,5,-0.52
5,2001,6,-0.73


---
## 3c · SPEI-3 at Chiang Mai (TerraClimate P & PET)

**Point:** Chiang Mai (18.7883°N, 98.9853°E)  
**Source:** `IDAHO_EPSCOR/TERRACLIMATE`  — monthly P (`pr`, mm) and PET (`pet`, 0.1 mm units)  

**SPEI-3 algorithm** (Vicente-Serrano et al. 2010):
1. Water-balance deficit `D = P − PET`
2. 3-month accumulation `D3 = D(t) + D(t-1) + D(t-2)`
3. For each calendar month: fit **3-parameter log-logistic** to D3 (via MLE, `scipy.stats.fisk`)
4. Transform CDF probabilities → standard normal Z-scores

In [11]:
def fetch_terraclimate_cm(year, month):
    """Monthly P (mm) and PET (mm) at Chiang Mai from TerraClimate."""
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, 'month')
    img   = (ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE')
               .filterDate(start, end)
               .select(['pr', 'pet'])
               .first())
    try:
        val = img.reduceRegion(
            reducer  = ee.Reducer.first(),
            geometry = CM_POINT,
            scale    = 4638       # TerraClimate native ~1/24 degree
        ).getInfo()
        pr  = float(val['pr'])  if val.get('pr')  is not None else np.nan
        pet = float(val['pet']) if val.get('pet') is not None else np.nan
        if not np.isnan(pet):
            pet = pet * 0.1       # TerraClimate PET stored in 0.1 mm units
        return pr, pet
    except Exception as e:
        print(f'  [warn] {year}-{month:02d}: {e}')
        return np.nan, np.nan


rows_tc = []
for year in YEARS:
    for month in MONTHS:
        pr, pet = fetch_terraclimate_cm(year, month)
        rows_tc.append({
            'year': year, 'month': month,
            'pr_mm':  round(pr,  2) if not np.isnan(pr)  else np.nan,
            'pet_mm': round(pet, 2) if not np.isnan(pet) else np.nan,
        })
    print(f'TerraClimate {year} done')

df_tc = pd.DataFrame(rows_tc)
print(f'\n{df_tc.head(6).to_string()}')

TerraClimate 2001 done
TerraClimate 2002 done
TerraClimate 2003 done
TerraClimate 2004 done
TerraClimate 2005 done
TerraClimate 2006 done
TerraClimate 2007 done
TerraClimate 2008 done
TerraClimate 2009 done
TerraClimate 2010 done
TerraClimate 2011 done
TerraClimate 2012 done
TerraClimate 2013 done
TerraClimate 2014 done
TerraClimate 2015 done
TerraClimate 2016 done
TerraClimate 2017 done
TerraClimate 2018 done
TerraClimate 2019 done
TerraClimate 2020 done
TerraClimate 2021 done
TerraClimate 2022 done
TerraClimate 2023 done

   year  month  pr_mm  pet_mm
0  2001      1    1.0   104.5
1  2001      2    1.0   114.1
2  2001      3   94.0   112.3
3  2001      4   14.0   187.8
4  2001      5  201.0   117.6
5  2001      6   93.0   116.8


In [12]:
def compute_spei3(df_tc):
    """
    Compute SPEI-3 from monthly P and PET.
    3-parameter log-logistic distribution fitted per calendar month (MLE).
    Reference: Vicente-Serrano et al. (2010) J. Climate 23:1696-1718.
    """
    df = df_tc.sort_values(['year', 'month']).reset_index(drop=True).copy()
    df['D']  = df['pr_mm'] - df['pet_mm']              # monthly water-balance deficit
    df['D3'] = df['D'].rolling(3, min_periods=3).sum() # 3-month accumulation

    spei3 = np.full(len(df), np.nan)

    for m in range(1, 13):
        idx   = df.index[df['month'] == m]
        valid = idx[df.loc[idx, 'D3'].notna()]
        if len(valid) < 4:
            continue
        vals = df.loc[valid, 'D3'].values

        # 3-parameter log-logistic: shift so all values > 0 (origin parameter gamma)
        gamma   = vals.min() - 1e-6
        shifted = vals - gamma

        # Fit log-logistic via MLE (fisk = log-logistic, loc fixed at 0)
        c, _, scale  = fisk.fit(shifted, floc=0)
        probs        = fisk.cdf(shifted, c, loc=0, scale=scale)
        probs        = np.clip(probs, 1e-6, 1 - 1e-6)
        spei3[valid] = norm.ppf(probs)

    df['spei3'] = np.round(spei3, 4)
    return df[['year', 'month', 'pr_mm', 'pet_mm', 'D', 'D3', 'spei3']]


df_spei = compute_spei3(df_tc)
print(df_spei[['year', 'month', 'pr_mm', 'pet_mm', 'D3', 'spei3']].head(12).to_string())

    year  month  pr_mm  pet_mm     D3   spei3
0   2001      1    1.0   104.5    NaN     NaN
1   2001      2    1.0   114.1    NaN     NaN
2   2001      3   94.0   112.3 -234.9  0.7177
3   2001      4   14.0   187.8 -305.2  0.3401
4   2001      5  201.0   117.6 -108.7  0.3915
5   2001      6   93.0   116.8 -114.2  0.1569
6   2001      7  151.0   105.9  104.7  0.2712
7   2001      8  229.0   110.7  139.6  0.1928
8   2001      9  174.0   109.4  228.0  0.2389
9   2001     10  182.0   107.6  257.3  0.3502
10  2001     11   23.0   101.8   60.2  0.1643
11  2001     12    6.0    80.0  -78.4  0.3080


---
## 4 · Merge Climate Data & Save All CSVs

In [13]:
# ── Merge CHIRPS + MEI v2 + SPEI-3 ────────────────────────────────────
df_climate = (df_rain
              .merge(df_mei,  on=['year', 'month'], how='left')
              .merge(df_spei[['year', 'month', 'spei3']], on=['year', 'month'], how='left'))

df_climate.to_csv('data_raw/climate_monthly.csv', index=False)

# ── Summary ────────────────────────────────────────────────────────────
print('=== Output Files (./data_raw/) ===')
print(f'{"File":<46}  {"Rows":>4}  Columns')
print('-' * 90)
for fname in sorted(os.listdir('data_raw')):
    if not fname.endswith('.csv'):
        continue
    fpath = os.path.join('data_raw', fname)
    tmp   = pd.read_csv(fpath)
    print(f'{fname:<46}  {len(tmp):>4}  {list(tmp.columns)}')

print('\n=== Climate preview (rows 1-6) ===')
print(df_climate.head(6).to_string())

=== Output Files (./data_raw/) ===
File                                            Rows  Columns
------------------------------------------------------------------------------------------
climate_monthly.csv                              276  ['year', 'month', 'chirps_mm', 'mei_v2', 'spei3']
lc_class_lookup.csv                               17  ['col', 'class_id', 'class_name']
mcd12q1_burned_by_lc_annual.csv                   23  ['year', 'lc01_burned_km2', 'lc02_burned_km2', 'lc03_burned_km2', 'lc04_burned_km2', 'lc05_burned_km2', 'lc06_burned_km2', 'lc07_burned_km2', 'lc08_burned_km2', 'lc09_burned_km2', 'lc10_burned_km2', 'lc11_burned_km2', 'lc12_burned_km2', 'lc13_burned_km2', 'lc14_burned_km2', 'lc15_burned_km2', 'lc16_burned_km2', 'lc17_burned_km2']
mcd64a1_monthly_burned_area.csv                  276  ['year', 'month', 'burned_area_km2']

=== Climate preview (rows 1-6) ===
   year  month  chirps_mm  mei_v2   spei3
0  2001      1    30.0799   -0.84     NaN
1  2001      2    13.77